# Kiểm tra kết quả Chương 3 — Khám phá và tiền xử lý dữ liệu HMEQ

Notebook này tính lại **mọi con số của Chương 3** (mục 3.2 và 3.3) trực tiếp từ `hmeq.csv` và tự động đối chiếu với bản thảo.
Mã được viết tường minh bằng pandas / scipy / scikit-learn / statsmodels (không phụ thuộc gói `creditscore`), để người đọc thấy từng phép tính.

| Phần | Mục trong đồ án | Nội dung |
|---|---|---|
| 0 | — | Nạp dữ liệu, hàm đối chiếu `check()` |
| 1 | 3.2.2 | Mô tả dữ liệu, Bảng 3.2 (giá trị khuyết) |
| 2 | 3.2.3 – Q1 | Mất cân bằng lớp; Accuracy của mô hình dự báo toàn 0 |
| 3 | 3.2.3 – Q2 | Phân phối, Shapiro–Wilk |
| 4 | 3.2.3 – Q3 | Tỷ lệ vỡ nợ theo JOB/REASON; tương quan Point-Biserial |
| 5 | 3.2.3 – Q4 | Hệ số φ; dendrogram; loại bỏ hồ sơ khuyết; Little's MCAR; AUC dự báo việc khuyết |
| 6 | 3.2.3 – Q5 | Tương quan Pearson; VIF |
| 7 | 3.3 | Chia dữ liệu 70/15/15; kiểm định chọn biến chỉ báo khuyết |
| 8 | — | Tổng hợp kết quả đối chiếu |

**Cách dùng:** `Runtime → Run all`, chọn file `hmeq.csv` khi hộp tải file hiện ra. Thời gian chạy dưới 1 phút.
Không cần cài thêm thư viện: Colab có sẵn mọi thư viện cần dùng.

## 0. Nạp dữ liệu và hàm đối chiếu
Chọn **một** cách nạp `hmeq.csv`: tải lên từ máy (mặc định) hoặc đọc từ Google Drive (`FROM_DRIVE = True`). Nếu đã giải nén gói `credit_scoring_colab.zip` ở notebook kia, file `credit_scoring/data/hmeq.csv` được dùng luôn.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from IPython.display import display

FROM_DRIVE = False
DRIVE_CSV = "/content/drive/MyDrive/hmeq.csv"
cands = [Path("hmeq.csv"), Path("/content/hmeq.csv"), Path("/content/credit_scoring/data/hmeq.csv"), Path("data/hmeq.csv"), Path("../data/hmeq.csv")]
path = next((p for p in cands if p.exists()), None)
if path is None:
    if FROM_DRIVE:
        from google.colab import drive; drive.mount("/content/drive"); path = Path(DRIVE_CSV)
    else:
        from google.colab import files; path = Path(next(iter(files.upload())))
df = pd.read_csv(path)

TARGET = "BAD"
NUM = ["LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG", "DELINQ", "CLAGE", "NINQ", "CLNO", "DEBTINC"]
CAT = ["REASON", "JOB"]
RAW = NUM + CAT
SEED = 42
print(path, df.shape)

# ---- hàm đối chiếu: so giá trị tính lại (làm tròn như bản thảo) với giá trị trong đồ án ----
RESULTS = []
def check(section, item, thesis, computed, dec=2, unit=""):
    """thesis: con số in trong đồ án; computed: giá trị tính lại; dec: số chữ số thập phân in trong đồ án."""
    c = round(float(computed), dec)
    ok = abs(c - float(thesis)) <= 10 ** (-dec) * 0.5 + 1e-12
    RESULTS.append({"Mục": section, "Nội dung": item, "Đồ án": f"{thesis:,.{dec}f}{unit}",
                    "Tính lại": f"{c:,.{dec}f}{unit}", "Khớp": "✓" if ok else "✗"})
    return ok

def show(section):
    t = pd.DataFrame([r for r in RESULTS if r["Mục"] == section])
    print(f"{section}: {(t['Khớp'] == '✓').sum()}/{len(t)} giá trị khớp")
    display(t.drop(columns="Mục").style.hide(axis="index"))

def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
print("Sẵn sàng.")

## 1. Mô tả bộ dữ liệu và Bảng 3.2 (mục 3.2.2)
5,960 hồ sơ, 13 biến; tỷ trọng các nhóm REASON, JOB; không có bản ghi trùng lặp; số lượng và tỷ lệ khuyết theo từng lớp.

In [ ]:
sec = "1. Mô tả dữ liệu (3.2.2)"
check(sec, "Số hồ sơ", 5960, len(df), 0)
check(sec, "Số biến", 13, df.shape[1], 0)
check(sec, "Số bản ghi trùng lặp", 0, df.duplicated().sum(), 0)

share = lambda c, v: (df[c] == v).mean() * 100          # tỷ trọng trên toàn bộ 5,960 hồ sơ (kể cả khuyết)
for c, v, t in [("REASON", "DebtCon", 65.9), ("REASON", "HomeImp", 29.9), ("JOB", "Other", 40.1),
                ("JOB", "ProfExe", 21.4), ("JOB", "Office", 15.9), ("JOB", "Mgr", 12.9)]:
    check(sec, f"{c} = {v} (%)", t, share(c, v), 1, "%")

# Bảng 3.2: số lượng khuyết, tỷ lệ khuyết, tỷ lệ khuyết trong từng lớp
miss = pd.DataFrame({
    "Số lượng khuyết": df[RAW].isna().sum(),
    "Tỷ lệ khuyết": df[RAW].isna().mean() * 100,
    "Tỷ lệ khuyết (BAD=0)": df.loc[df.BAD == 0, RAW].isna().mean() * 100,
    "Tỷ lệ khuyết (BAD=1)": df.loc[df.BAD == 1, RAW].isna().mean() * 100,
}).sort_values("Tỷ lệ khuyết", ascending=False)
display(miss.round(2))

T32 = {"DEBTINC": (1267, 21.26, 10.08, 66.11), "DEROG": (708, 11.88, 13.02, 7.32), "DELINQ": (580, 9.73, 10.65, 6.06),
       "MORTDUE": (518, 8.69, 8.64, 8.92), "YOJ": (515, 8.64, 9.43, 5.47), "NINQ": (510, 8.56, 9.12, 6.31),
       "CLAGE": (308, 5.17, 4.82, 6.56), "JOB": (279, 4.68, 5.37, 1.93), "REASON": (252, 4.23, 4.28, 4.04),
       "CLNO": (222, 3.72, 3.54, 4.46), "VALUE": (112, 1.88, 0.15, 8.83), "LOAN": (0, 0, 0, 0)}
for v, (n, r, r0, r1) in T32.items():
    check(sec, f"Bảng 3.2 – {v}: số khuyết", n, miss.loc[v, "Số lượng khuyết"], 0)
    check(sec, f"Bảng 3.2 – {v}: tỷ lệ khuyết", r, miss.loc[v, "Tỷ lệ khuyết"], 2, "%")
    check(sec, f"Bảng 3.2 – {v}: khuyết khi BAD=0", r0, miss.loc[v, "Tỷ lệ khuyết (BAD=0)"], 2, "%")
    check(sec, f"Bảng 3.2 – {v}: khuyết khi BAD=1", r1, miss.loc[v, "Tỷ lệ khuyết (BAD=1)"], 2, "%")
check(sec, "BAD: số khuyết", 0, df[TARGET].isna().sum(), 0)
show(sec)

## 2. Q1 – Mất cân bằng lớp (mục 3.2.3)
Ngoài tỷ trọng hai lớp, ô này tính Accuracy, Recall, F1 của một **bộ dự báo giả định gán mọi hồ sơ là không vỡ nợ** (`DummyClassifier(strategy="most_frequent")`), là ví dụ dùng ở mục 2.4 và 3.2.3: Accuracy = 80.05% nhưng không nhận diện được hồ sơ vỡ nợ nào.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score
sec = "2. Q1 – Mất cân bằng lớp"
vc = df[TARGET].value_counts().sort_index()
display(pd.DataFrame({"Số hồ sơ": vc, "Tỷ trọng (%)": (vc / len(df) * 100).round(2)}))
check(sec, "BAD = 0: số hồ sơ", 4771, vc[0], 0)
check(sec, "BAD = 0: tỷ trọng", 80.05, vc[0] / len(df) * 100, 2, "%")
check(sec, "BAD = 1: số hồ sơ", 1189, vc[1], 0)
check(sec, "BAD = 1: tỷ trọng", 19.95, vc[1] / len(df) * 100, 2, "%")

y = df[TARGET].values
dummy = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y), 1)), y)
yhat = dummy.predict(np.zeros((len(y), 1)))                   # = 0 cho mọi hồ sơ
check(sec, "Dự báo toàn 0: Accuracy", 80.05, accuracy_score(y, yhat) * 100, 2, "%")
check(sec, "Dự báo toàn 0: Recall lớp vỡ nợ", 0, recall_score(y, yhat), 2)
check(sec, "Dự báo toàn 0: F1 lớp vỡ nợ", 0, f1_score(y, yhat, zero_division=0), 2)
show(sec)

## 3. Q2 – Phân phối của các biến (mục 3.2.3)
Thống kê mô tả, độ lệch (skewness) và kiểm định Shapiro–Wilk trên giá trị quan sát được của từng biến định lượng. Với n > 5,000, scipy cảnh báo p-value có thể kém chính xác; kết luận không đổi vì p nhỏ hơn 0.001 nhiều bậc.

In [ ]:
sec = "3. Q2 – Phân phối"
desc = df[NUM].describe().T
desc["skew"] = df[NUM].skew()
sw = {c: stats.shapiro(df[c].dropna()) for c in NUM}
desc["Shapiro W"] = [sw[c].statistic for c in NUM]
desc["Shapiro p"] = [sw[c].pvalue for c in NUM]
display(desc.round(3))
check(sec, "DEBTINC: trung bình", 33.78, df.DEBTINC.mean(), 2)
check(sec, "DEBTINC: giá trị lớn nhất", 203.31, df.DEBTINC.max(), 2)
check(sec, "Số biến lệch phải (skew > 0)", 10, (desc["skew"] > 0).sum(), 0)
check(sec, "Số biến bị Shapiro–Wilk bác bỏ chuẩn (p < 0.001)", 10, (desc["Shapiro p"] < 0.001).sum(), 0)
show(sec)

## 4. Q3 – Các biến phân biệt khách hàng vỡ nợ (mục 3.2.3)
Tỷ lệ vỡ nợ theo JOB và REASON (Hình 3.2, 3.3) và hệ số Point-Biserial $r_{pb}$ với BAD (Bảng 3.3). Mức ý nghĩa: *** p < 0.001, ** p < 0.01, * p < 0.05 (giống Bảng 3.4).

In [ ]:
import matplotlib.pyplot as plt
sec = "4. Q3 – Biến phân biệt"
for c in CAT:
    t = df.groupby(c)[TARGET].agg(["size", "mean"]).sort_values("mean", ascending=False)
    t["mean"] = (t["mean"] * 100).round(1); display(t.rename(columns={"size": "Số hồ sơ", "mean": "Tỷ lệ vỡ nợ (%)"}))
br = lambda c, v: df.loc[df[c] == v, TARGET].mean() * 100
for c, v, t in [("JOB", "Sales", 34.9), ("JOB", "Self", 30.1), ("JOB", "Office", 13.2), ("JOB", "ProfExe", 16.6),
                ("JOB", "Mgr", 23.3), ("JOB", "Other", 23.2), ("REASON", "HomeImp", 22.2), ("REASON", "DebtCon", 19.0)]:
    check(sec, f"Tỷ lệ vỡ nợ {c} = {v}", t, br(c, v), 1, "%")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, c in zip(axes, CAT):
    pd.crosstab(df[c], df[TARGET]).plot.bar(ax=ax, rot=0); ax.set_title(c); ax.legend(["BAD=0", "BAD=1"])
plt.tight_layout(); plt.show()

# Point-Biserial: Pearson giữa biến định lượng (giá trị quan sát được) và BAD
rows = []
for c in NUM:
    x = df[[c, TARGET]].dropna()
    r, p = stats.pointbiserialr(x[TARGET], x[c])
    rows.append({"Biến": c, "r_pb": r, "p-value": p, "Mức ý nghĩa": stars(p)})
rpb = pd.DataFrame(rows).sort_values("r_pb", ascending=False).set_index("Biến")
display(rpb.round(4))
T33 = {"DELINQ": (0.354, "***"), "DEROG": (0.276, "***"), "DEBTINC": (0.200, "***"), "NINQ": (0.175, "***"),
       "CLAGE": (-0.170, "***"), "LOAN": (-0.075, "***"), "YOJ": (-0.060, "***"), "MORTDUE": (-0.048, "***"),
       "VALUE": (-0.030, "*"), "CLNO": (-0.004, "")}
for v, (r, s) in T33.items():
    ok = check(sec, f"Bảng 3.3 – r_pb {v} {s}", r, rpb.loc[v, "r_pb"], 3)
    if rpb.loc[v, "Mức ý nghĩa"] != s:
        print(f"Lưu ý: {v} có mức ý nghĩa '{rpb.loc[v, 'Mức ý nghĩa']}' (p = {rpb.loc[v, 'p-value']:.4f}), đồ án ghi '{s}'")
show(sec)

## 5. Q4 – Liên hệ giữa dữ liệu khuyết và trạng thái vỡ nợ (mục 3.2.3)
### 5a. Hệ số φ giữa chỉ báo khuyết và BAD (Bảng 3.4)
$M_j = 1$ nếu biến $j$ khuyết. φ là hệ số Pearson giữa hai biến nhị phân; p-value từ kiểm định t của hệ số tương quan (`scipy.stats.pearsonr`); kiểm định khi bình phương 2×2 không hiệu chỉnh Yates cho p gần như trùng (sai khác ở chữ số thứ tư).

In [ ]:
sec = "5a. Q4 – Hệ số φ"
rows = []
for c in RAW:
    m = df[c].isna().astype(int)
    if m.sum() == 0:
        continue
    phi, p = stats.pearsonr(m, df[TARGET])          # φ = hệ số Pearson giữa hai biến nhị phân
    rows.append({"Biến": c, "φ": phi, "p-value": p, "Mức ý nghĩa": stars(p)})
phi = pd.DataFrame(rows).sort_values("φ", ascending=False).set_index("Biến")
display(phi.round(3))
T34 = {"DEBTINC": (0.547, 0.000), "VALUE": (0.256, 0.000), "CLAGE": (0.031, 0.015), "CLNO": (0.019, 0.136),
       "MORTDUE": (0.004, 0.760), "REASON": (-0.005, 0.714), "NINQ": (-0.040, 0.002), "YOJ": (-0.056, 0.000),
       "DELINQ": (-0.062, 0.000), "JOB": (-0.065, 0.000), "DEROG": (-0.070, 0.000)}
for v, (f, p) in T34.items():
    check(sec, f"Bảng 3.4 – φ {v}", f, phi.loc[v, "φ"], 3)
    check(sec, f"Bảng 3.4 – p-value {v}", p, phi.loc[v, "p-value"], 3)
show(sec)

### 5b. Cấu trúc khuyết đồng thời — dendrogram (Hình 3.5)
Ma trận chỉ báo khuyết $M$ (biến × hồ sơ). Khoảng cách Euclid giữa hai biến $a, b$ bằng $\sqrt{n_{10}+n_{01}}$ (số hồ sơ chỉ khuyết một trong hai biến); gộp cụm theo phương pháp Ward — giống mã gốc của đồ án (`linkage(M, method="ward", metric="euclidean")`). Bảng in kèm độ cao của từng lần gộp.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
sec = "5b. Q4 – Dendrogram"
def merges(d):
    cols = [c for c in RAW if d[c].isna().any()]
    Z = linkage(d[cols].isna().astype(int).T.values, method="ward", metric="euclidean")
    names = list(cols)
    for a, b, h, _ in Z:
        names.append(names[int(a)] + " + " + names[int(b)])
    return Z, cols, pd.DataFrame({"Cụm mới": names[len(cols):], "Độ cao gộp": Z[:, 2].round(1)})

fig, axes = plt.subplots(2, 1, figsize=(9, 7))
out = {}
for ax, (lab, d) in zip(axes, [("Toàn bộ mẫu", df), ("Chỉ nhóm vỡ nợ (BAD = 1)", df[df.BAD == 1])]):
    Z, cols, tab = merges(d); out[lab] = tab
    dendrogram(Z, labels=cols, ax=ax, leaf_rotation=90); ax.set_title(lab); ax.set_ylabel("Khoảng cách (Ward)")
plt.tight_layout(); plt.show()
for k, t in out.items():
    print(k); display(t)

full, bad = out["Toàn bộ mẫu"], out["Chỉ nhóm vỡ nợ (BAD = 1)"]
h = lambda t, s: t.loc[t["Cụm mới"] == s, "Độ cao gộp"].iloc[0]
check(sec, "Toàn mẫu: CLAGE + CLNO gộp đầu tiên", 9.3, h(full, "CLAGE + CLNO"), 1)
check(sec, "Toàn mẫu: VALUE + REASON", 18.0, h(full, "VALUE + REASON"), 1)
check(sec, "Toàn mẫu: DEBTINC gộp cuối cùng", 48.0, full["Độ cao gộp"].iloc[-1], 1)
# nhóm vỡ nợ: CLAGE, CLNO, DELINQ, NINQ tạo thành một cụm trước khi có biến nào khác gia nhập
first4 = bad["Cụm mới"].iloc[2]
check(sec, "BAD=1: cụm thứ 3 gồm đúng {CLAGE, CLNO, DELINQ, NINQ} (1 = đúng)", 1,
      set(first4.split(" + ")) == {"CLAGE", "CLNO", "DELINQ", "NINQ"}, 0)
show(sec)

### 5c. Hệ quả của việc loại bỏ hồ sơ khuyết (Bảng 3.5, 3.6)

In [ ]:
sec = "5c. Q4 – Loại bỏ hồ sơ khuyết"
nmiss = df[RAW].isna().sum(axis=1)
cc, inc = df[nmiss == 0], df[nmiss > 0]
B = df[TARGET].sum()
t35 = pd.DataFrame([[len(g), len(g) / len(df) * 100, g[TARGET].sum(), g[TARGET].sum() / B * 100, g[TARGET].mean() * 100]
                    for g in (df, cc, inc)], index=["Toàn bộ", "Hồ sơ đầy đủ", "Có ≥ 1 giá trị khuyết"],
                   columns=["Số hồ sơ", "Tỷ trọng %", "Số vỡ nợ", "% số vỡ nợ", "Tỷ lệ vỡ nợ %"])
display(t35.round(1))
for lab, vals in {"Toàn bộ": (5960, 100.0, 1189, 100.0, 19.9), "Hồ sơ đầy đủ": (3364, 56.4, 300, 25.2, 8.9),
                  "Có ≥ 1 giá trị khuyết": (2596, 43.6, 889, 74.8, 34.2)}.items():
    for col, v in zip(t35.columns, vals):
        check(sec, f"Bảng 3.5 – {lab}: {col}", v, t35.loc[lab, col], 0 if col in ("Số hồ sơ", "Số vỡ nợ") else 1)

t36 = df.groupby(nmiss.clip(upper=4))[TARGET].agg(["size", "mean"])
t36["Tỷ trọng %"] = t36["size"] / len(df) * 100; t36["mean"] *= 100
t36.index = ["0 (đầy đủ)", "1", "2", "3", "≥ 4"]
display(t36.rename(columns={"size": "Số hồ sơ", "mean": "Tỷ lệ vỡ nợ %"}).round(1))
for lab, (n, s, r) in {"0 (đầy đủ)": (3364, 56.4, 8.9), "1": (1589, 26.7, 38.1), "2": (449, 7.5, 40.1),
                       "3": (219, 3.7, 17.8), "≥ 4": (339, 5.7, 18.9)}.items():
    check(sec, f"Bảng 3.6 – {lab} trường khuyết: số hồ sơ", n, t36.loc[lab, "size"], 0)
    check(sec, f"Bảng 3.6 – {lab} trường khuyết: tỷ trọng", s, t36.loc[lab, "Tỷ trọng %"], 1, "%")
    check(sec, f"Bảng 3.6 – {lab} trường khuyết: tỷ lệ vỡ nợ", r, t36.loc[lab, "mean"], 1, "%")
show(sec)

### 5d. Kiểm định Little's MCAR (Little, 1988)
Trên 10 biến định lượng (chuẩn hóa để ổn định số học). Trung bình $\hat\mu$ và hiệp phương sai $\hat\Sigma$ ước lượng bằng thuật toán EM dưới giả định chuẩn đa biến. Với mỗi mẫu khuyết $j$ ($n_j$ hồ sơ, tập biến quan sát $o_j$):

$$d^2=\sum_j n_j(\bar{\mathbf y}_{o_j}-\hat{\boldsymbol\mu}_{o_j})^\top\hat\Sigma_{o_j o_j}^{-1}(\bar{\mathbf y}_{o_j}-\hat{\boldsymbol\mu}_{o_j}),\qquad df=\sum_j |o_j| - p.$$

Ô này chạy trong vài giây đến 1 phút.

In [ ]:
sec = "5d. Q4 – Little's MCAR"
def em_mvn(X, iters=200, tol=1e-6):
    """EM cho trung bình và hiệp phương sai của phân phối chuẩn đa biến khi có giá trị khuyết."""
    mu = np.nanmean(X, 0); S = np.diag(np.nanvar(X, 0))
    for _ in range(iters):
        Xh, C = X.copy(), np.zeros_like(S)
        for i, row in enumerate(X):
            m = np.isnan(row)
            if not m.any():
                continue
            o = ~m
            B_ = S[np.ix_(m, o)] @ np.linalg.pinv(S[np.ix_(o, o)])
            Xh[i, m] = mu[m] + B_ @ (row[o] - mu[o])                  # kỳ vọng có điều kiện (bước E)
            C[np.ix_(m, m)] += S[np.ix_(m, m)] - B_ @ S[np.ix_(o, m)]  # hiệp phương sai có điều kiện
        mu_new = Xh.mean(0); D = Xh - mu_new; S_new = (D.T @ D + C) / len(X)   # bước M
        done = np.max(np.abs(mu_new - mu)) < tol and np.max(np.abs(S_new - S)) < tol
        mu, S = mu_new, S_new
        if done:
            break
    return mu, S

def little_mcar(Xdf):
    X = Xdf.to_numpy(float)
    Z = (X - np.nanmean(X, 0)) / np.nanstd(X, 0)
    mu, Sg = em_mvn(Z)
    pat = pd.Series([tuple(r) for r in np.isnan(Z)])
    d2, dof = 0.0, 0
    for p, idx in pat.groupby(pat).groups.items():
        o = ~np.array(p)
        if not o.any():
            continue
        diff = Z[np.asarray(idx)][:, o].mean(0) - mu[o]
        d2 += len(idx) * diff @ np.linalg.pinv(Sg[np.ix_(o, o)]) @ diff
        dof += o.sum()
    dof -= Z.shape[1]
    return d2, dof, stats.chi2.sf(d2, dof), pat.nunique()

d2, dof, p_mcar, npat = little_mcar(df[NUM])
print(f"d² = {d2:,.1f}, df = {dof}, p = {p_mcar:.3g}, số mẫu khuyết = {npat}")
check(sec, "Số mẫu khuyết khác nhau", 75, npat, 0)
check(sec, "Thống kê d²", 4399.3, d2, 1)
check(sec, "Bậc tự do", 478, dof, 0)
check(sec, "p < 0.001 (1 = đúng)", 1, p_mcar < 0.001, 0)
show(sec)

### 5e. Tính dự báo và giá trị thông tin của dữ liệu khuyết (Bảng 3.7)
Với mỗi biến $j$ có khuyết: hồi quy logistic dự báo chỉ báo $M_j$ từ các biến còn lại (điền trung vị, one-hot biến định tính kèm nhóm khuyết) và chỉ báo khuyết của chúng; AUC tính trên dự báo ngoài fold của 5-fold Stratified CV (`random_state=42`). Lặp lại sau khi thêm BAD làm biến đầu vào. Ô này chạy trong vài giây.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
sec = "5e. Q4 – AUC dự báo việc khuyết"
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for c in [v for v in RAW if df[v].isna().any()]:
    m = df[c].isna().astype(int)
    others = [v for v in RAW if v != c]
    Xo = df[others].assign(**{v: df[v].fillna(df[v].median()) for v in others if v in NUM})
    Xo = pd.get_dummies(Xo, columns=[v for v in others if v in CAT], dummy_na=True, dtype=float)
    Xo = Xo.join(df[others].isna().add_suffix("_na").astype(float))
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    p_x = cross_val_predict(lr, Xo, m, cv=skf, method="predict_proba")[:, 1]
    p_xy = cross_val_predict(lr, Xo.assign(BAD=df[TARGET]), m, cv=skf, method="predict_proba")[:, 1]
    rows.append({"Biến": c, "% khuyết": m.mean() * 100, "AUC (biến khác)": roc_auc_score(m, p_x),
                 "AUC (+ BAD)": roc_auc_score(m, p_xy),
                 "Vỡ nợ khi khuyết %": df.loc[m == 1, TARGET].mean() * 100,
                 "Vỡ nợ khi có %": df.loc[m == 0, TARGET].mean() * 100})
mar = pd.DataFrame(rows).sort_values("% khuyết", ascending=False).set_index("Biến")
mar.insert(3, "Mức tăng", mar["AUC (+ BAD)"] - mar["AUC (biến khác)"])
display(mar.round(3))
T37 = {"DEBTINC": (0.714, 0.794, 62.0, 8.6), "DEROG": (0.947, 0.955, 12.3, 21.0), "DELINQ": (0.983, 0.981, 12.4, 20.8),
       "MORTDUE": (0.894, 0.895, 20.5, 19.9), "YOJ": (0.828, 0.828, 12.6, 20.6), "NINQ": (0.948, 0.945, 14.7, 20.4),
       "CLAGE": (0.995, 0.995, 25.3, 19.7), "JOB": (0.879, 0.886, 8.2, 20.5), "REASON": (0.788, 0.786, 19.0, 20.0),
       "CLNO": (1.000, 1.000, 23.9, 19.8), "VALUE": (0.844, 0.938, 93.8, 18.5)}
for v, (a0, a1, b1, b0) in T37.items():
    check(sec, f"Bảng 3.7 – {v}: AUC biến khác", a0, mar.loc[v, "AUC (biến khác)"], 3)
    check(sec, f"Bảng 3.7 – {v}: AUC + BAD", a1, mar.loc[v, "AUC (+ BAD)"], 3)
    check(sec, f"Bảng 3.7 – {v}: vỡ nợ khi khuyết", b1, mar.loc[v, "Vỡ nợ khi khuyết %"], 1, "%")
    check(sec, f"Bảng 3.7 – {v}: vỡ nợ khi có", b0, mar.loc[v, "Vỡ nợ khi có %"], 1, "%")
show(sec)

## 6. Q5 – Đa cộng tuyến (mục 3.2.3)
Tương quan Pearson (trên các cặp quan sát được) và VIF. $VIF_i = 1/(1-R_i^2)$, với $R_i^2$ từ hồi quy $X_i$ theo các biến còn lại **có hệ số chặn**.

Ô này tính VIF theo hai cách để đối chiếu: (A) đúng định nghĩa, có hệ số chặn; (B) như mã gốc của notebook đồ án — `variance_inflation_factor(X.values, i)` **không** `add_constant`, trên 9 biến (không có NINQ), 3,551 hồ sơ đầy đủ. Cách (B) cho $R^2$ không định tâm (uncentered), nên VIF bị phóng đại với các biến có giá trị trung bình lớn.

In [ ]:
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
sec = "6. Q5 – Đa cộng tuyến"
corr = df[NUM].corr()
plt.figure(figsize=(7, 6)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1); plt.show()
check(sec, "r(MORTDUE, VALUE)", 0.88, corr.loc["MORTDUE", "VALUE"], 2)
check(sec, "r(LOAN, VALUE)", 0.34, corr.loc["LOAN", "VALUE"], 2)
check(sec, "r(CLNO, MORTDUE)", 0.32, corr.loc["CLNO", "MORTDUE"], 2)

V9 = [c for c in NUM if c != "NINQ"]
X9 = df[V9].dropna()
vif = pd.DataFrame(index=V9)
vif["(A) có hệ số chặn"] = [variance_inflation_factor(add_constant(X9).values, i + 1) for i in range(len(V9))]
vif["(B) không hệ số chặn – mã gốc"] = [variance_inflation_factor(X9.values, i) for i in range(len(V9))]
X10 = df[NUM].dropna()
vif.loc[:, "(A) 10 biến, có hệ số chặn"] = pd.Series(
    [variance_inflation_factor(add_constant(X10).values, i + 1) for i in range(len(NUM))], index=NUM)
print(f"Số hồ sơ đầy đủ trên 9 biến: {len(X9):,}; trên 10 biến: {len(X10):,}")
display(vif.sort_values("(B) không hệ số chặn – mã gốc", ascending=False).round(2))

# statsmodels >= 0.14 có thể tự thêm hệ số chặn khi tính R^2; tính tay để chắc chắn cách (B) là không định tâm
def vif_uncentered(X):
    X = X.to_numpy(float); out = []
    for i in range(X.shape[1]):
        y_, Z_ = X[:, i], np.delete(X, i, 1)
        b = np.linalg.lstsq(Z_, y_, rcond=None)[0]
        out.append(1 / (((y_ - Z_ @ b) ** 2).sum() / (y_ ** 2).sum()))
    return pd.Series(out, index=V9)
vif["(B) tính tay"] = vif_uncentered(X9)
display(vif[["(B) không hệ số chặn – mã gốc", "(B) tính tay"]].round(2))

TV = {"VALUE": 20.90, "MORTDUE": 16.87, "DEBTINC": 9.27, "CLNO": 7.46, "CLAGE": 6.05, "LOAN": 4.65, "YOJ": 2.63,
      "DELINQ": 1.17, "DEROG": 1.11}
for v, t in TV.items():
    check(sec, f"VIF {v} – so với cách (B), như đồ án", t, vif.loc[v, "(B) tính tay"], 2)
for v in ["VALUE", "MORTDUE"]:
    print(f"{v}: VIF đúng định nghĩa (có hệ số chặn) = {vif.loc[v, '(A) có hệ số chặn']:.2f} < 10")
show(sec)

> **Lưu ý về VIF:** các giá trị VIF trong đồ án (VALUE 20.90, MORTDUE 16.87, …) tái lập được **chính xác** bằng cách (B), nhưng cách (B) bỏ hệ số chặn nên không đúng định nghĩa VIF. Tính đúng (có hệ số chặn), VALUE và MORTDUE có VIF khoảng 4.4, **dưới ngưỡng 10**; tương quan r = 0.88 giữa hai biến vẫn đúng và vẫn là cặp tương quan cao nhất.

## 7. Tiền xử lý (mục 3.3): chia dữ liệu và chọn biến chỉ báo khuyết
Chia phân tầng 70/15/15 bằng `train_test_split` hai lần (`random_state=42`), giống `creditscore.data.split_data`. Trên **tập huấn luyện**, mỗi biến có khuyết được kiểm định bằng hồi quy logistic $P(\text{BAD}=1)=\sigma(\beta_0+\beta_1 M_j)$; biến chỉ báo được giữ khi p-value của $\beta_1$ < 0.05; $\text{OR}=e^{\beta_1}$.

In [ ]:
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
sec = "7. Tiền xử lý (3.3)"
train, temp = train_test_split(df, test_size=0.30, stratify=df[TARGET], random_state=SEED)
valid, test = train_test_split(temp, test_size=0.50, stratify=temp[TARGET], random_state=SEED)
sizes = {"Huấn luyện": train, "Kiểm định": valid, "Kiểm tra": test}
display(pd.DataFrame({k: [len(v), v[TARGET].sum(), v[TARGET].mean() * 100] for k, v in sizes.items()},
                     index=["Số hồ sơ", "Số vỡ nợ", "Tỷ lệ vỡ nợ %"]).round(2))
for k, n in zip(sizes, (4172, 894, 894)):
    check(sec, f"Số hồ sơ tập {k.lower()}", n, len(sizes[k]), 0)

rows = []
for c in RAW:
    m = train[c].isna().astype(float)
    if m.sum() == 0:
        continue
    res = sm.Logit(train[TARGET].astype(float), sm.add_constant(m)).fit(disp=0)
    rows.append({"Biến": c, "Số khuyết (train)": int(m.sum()), "β1": res.params.iloc[1],
                 "OR": np.exp(res.params.iloc[1]), "p-value": res.pvalues.iloc[1], "Giữ chỉ báo": res.pvalues.iloc[1] < 0.05})
ind = pd.DataFrame(rows).sort_values("p-value").set_index("Biến")
display(ind.round(4))
check(sec, "Số biến chỉ báo khuyết được giữ (trên 11)", 8, ind["Giữ chỉ báo"].sum(), 0)
check(sec, "OR của chỉ báo DEBTINC", 17.79, ind.loc["DEBTINC", "OR"], 2)
check(sec, "OR của chỉ báo VALUE", 49.46, ind.loc["VALUE", "OR"], 2)
print("Giữ:", list(ind.index[ind["Giữ chỉ báo"]]), "| Bỏ:", list(ind.index[~ind["Giữ chỉ báo"]]))
show(sec)

## 8. Tổng hợp
Mọi dòng ✗ (nếu có) được liệt kê ở bảng thứ hai.

In [ ]:
R = pd.DataFrame(RESULTS)
summ = R.groupby("Mục", sort=False)["Khớp"].agg(Khớp=lambda s: (s == "✓").sum(), Tổng="size")
display(summ)
print(f"TỔNG: {(R['Khớp'] == '✓').sum()}/{len(R)} giá trị khớp với đồ án")
bad = R[R["Khớp"] != "✓"]
display(bad if len(bad) else "Không có sai khác.")